# NIFTY 50 Statistical Pairs Trading

A systematic cointegration-based pairs trading pipeline on the NIFTY 50 universe.

**Strategy Logic:**
- Scan all NIFTY 50 pairs for cointegration using Engle-Granger test
- Filter pairs by rolling beta stability and half-life of mean reversion
- Trade the spread using z-score signals (entry ±2σ, exit 0.5σ)
- Use rolling OLS hedge ratio to adapt to regime changes

**Universe:** NIFTY 50 (46 stocks after liquidity filter)  
**Period:** 2018-01-01 to 2026-03-10  
**Data:** Yahoo Finance via yfinance  

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from statsmodels.regression.rolling import RollingOLS
from itertools import combinations

# Global parameters
START_DATE = '2018-01-01'
END_DATE = '2026-03-10'
RISK_FREE_RATE = 0.06      # India RBI repo rate
ENTRY_ZSCORE = 2.0         # Entry threshold
EXIT_ZSCORE = 0.5          # Exit threshold
ROLLING_WINDOW = 60        # Rolling beta and zscore window
TRANSACTION_COST = 0.001   # 10bps per leg
MAX_BETA_STD = 0.2         # Max rolling beta std for pair filter
MIN_COVERAGE = 0.75        # Min data coverage for ticker inclusion

## 2. Data Pipeline

In [ ]:
def get_prices(tickers, start, end, min_coverage=MIN_COVERAGE):
    """
    Download adjusted close prices for a list of tickers.
    Drops tickers with insufficient history, forward-fills gaps.
    """
    data = yf.download(tickers, start=start, end=end, auto_adjust=True)
    data = data['Close']

    # Drop tickers with less than min_coverage of the date range
    threshold = int(len(data) * min_coverage)
    data.dropna(axis=1, thresh=threshold, inplace=True)

    # Forward fill remaining gaps, drop any unfillable rows
    data.ffill(inplace=True)
    data.dropna(inplace=True)

    dropped = set(tickers) - set(data.columns)
    if dropped:
        print(f"Dropped tickers with insufficient history: {dropped}")

    return data

In [ ]:
# Fetch NIFTY 50 constituents from Wikipedia
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://en.wikipedia.org/wiki/NIFTY_50"
tables = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
symbols = tables[1]['Symbol'].tolist()
tickers = [s + ".NS" for s in symbols]
print(f"NIFTY 50 tickers loaded: {len(tickers)}")

In [ ]:
# Download prices for all NIFTY 50 tickers
ticker_prices = get_prices(tickers, START_DATE, END_DATE)
valid_tickers = ticker_prices.columns.tolist()

print(f"Valid tickers: {len(valid_tickers)}")
print(f"Date range: {ticker_prices.index[0].date()} to {ticker_prices.index[-1].date()}")
print(f"Trading days: {len(ticker_prices)}")

## 3. Cointegration Testing

We use the Engle-Granger test to identify cointegrated pairs. The null hypothesis is that no cointegration exists (unit root in the spread). We reject at p < 0.05.

To account for the asymmetry of Engle-Granger (order of variables matters), we run the test both ways and take the more conservative p-value (maximum).

In [ ]:
def test_cointegration(s1, s2, threshold=0.05):
    """
    Engle-Granger cointegration test run in both directions.
    Returns (is_cointegrated, conservative_p_value).
    """
    s1 = np.array(s1.dropna())
    s2 = np.array(s2.dropna())

    if len(s1) == 0 or len(s2) == 0 or len(s1) != len(s2):
        return False, np.nan

    _, pval1, _ = coint(s1, s2)
    _, pval2, _ = coint(s2, s1)

    # Take conservative (max) p-value — only declare cointegration if both directions agree
    p_val = max(pval1, pval2)
    return p_val < threshold, p_val

In [ ]:
# Scan all pairs for cointegration
all_pairs = list(combinations(valid_tickers, 2))
print(f"Total pairs to test: {len(all_pairs)}")

results = []
for pair in all_pairs:
    is_coint, p_val = test_cointegration(ticker_prices[pair[0]], ticker_prices[pair[1]])
    results.append((pair[0], pair[1], is_coint, p_val))

results_df = pd.DataFrame(results, columns=['Ticker1', 'Ticker2', 'Is_Coint', 'P_Value'])
results_df.to_csv('cointegration_results.csv', index=False)

cointegrated_pairs = results_df[results_df['Is_Coint']].sort_values('P_Value')
print(f"\nCointegrated pairs: {len(cointegrated_pairs)} out of {len(all_pairs)}")
print(cointegrated_pairs.head(10).to_string())

## 4. Pair Filtering

Passing the cointegration test is necessary but not sufficient. We apply two additional filters:

1. **Rolling beta stability** — the hedge ratio must be stable over time (std < 0.2). An unstable beta means the relationship is not reliably tradeable.
2. **Half-life of mean reversion** — measures how quickly the spread reverts to its mean. Computed from AR(1) regression: `half-life = -log(2) / log(β)`. Shorter is better for EOD trading.

In [ ]:
def compute_rolling_beta(y, x, window=ROLLING_WINDOW):
    """
    Rolling OLS hedge ratio. Returns a Series of daily betas.
    Uses price levels — not returns.
    """
    x_const = sm.add_constant(x)
    model = RollingOLS(y, x_const, window=window)
    results = model.fit()
    return results.params.iloc[:, 1]  # slope coefficient, dynamically indexed


def compute_halflife(spread):
    """
    Half-life of mean reversion via AR(1) regression.
    half-life = -log(2) / log(beta) where beta is the AR(1) coefficient.
    """
    Y = np.array(spread[1:])
    X = sm.add_constant(np.array(spread[:-1]))
    beta = sm.OLS(Y, X).fit().params[1]
    return -np.log(2) / np.log(beta)

In [ ]:
# Compute beta stability and half-life for all cointegrated pairs
filter_results = []
for _, row in cointegrated_pairs.iterrows():
    pair = (row['Ticker1'], row['Ticker2'])
    try:
        rolling_beta = compute_rolling_beta(ticker_prices[pair[1]], ticker_prices[pair[0]]).dropna()
        beta_std = rolling_beta.std()
        spread = (ticker_prices[pair[1]] - rolling_beta * ticker_prices[pair[0]]).dropna()
        spread = spread.replace([np.inf, -np.inf], np.nan).dropna()
        halflife = compute_halflife(spread)
        filter_results.append({
            'pair': f"{pair[0]}/{pair[1]}",
            'p_value': row['P_Value'],
            'beta_std': beta_std,
            'halflife': halflife
        })
    except:
        pass

filter_df = pd.DataFrame(filter_results)

# Apply filters
stable_pairs = filter_df[filter_df['beta_std'] < MAX_BETA_STD].sort_values('halflife')
print(f"Pairs with stable beta (std < {MAX_BETA_STD}): {len(stable_pairs)}")
print(stable_pairs.to_string())

## 5. Signal Generation

Signals are generated from the z-score of the rolling spread:
- **Z > +2** → Short spread (sell leg1, buy leg2)
- **Z < -2** → Long spread (buy leg1, sell leg2)
- **|Z| < 0.5** → Exit position

Positions persist via forward-fill until exit condition fires.

In [ ]:
def compute_zscore(spread, window=ROLLING_WINDOW):
    """
    Rolling z-score of spread. Uses only past data — no lookahead bias.
    """
    rolling_mean = spread.rolling(window=window).mean()
    rolling_std = spread.rolling(window=window).std()
    return (spread - rolling_mean) / rolling_std


def generate_signals(zscore, entry=ENTRY_ZSCORE, exit_threshold=EXIT_ZSCORE):
    """
    Generate trading signals from z-score.
    Returns DataFrame with signal column: 1 (long), -1 (short), 0 (flat).
    """
    signals = pd.DataFrame(index=zscore.index)
    signals['signal'] = np.nan
    signals['entry_short'] = (zscore > entry).astype(int)
    signals['entry_long'] = (zscore < -entry).astype(int)
    signals['exit'] = (zscore.abs() < exit_threshold).astype(int)

    signals.loc[signals['entry_short'] == 1, 'signal'] = -1
    signals.loc[signals['entry_long'] == 1, 'signal'] = 1
    signals.loc[signals['exit'] == 1, 'signal'] = 0

    # Forward-fill to hold position until exit fires
    signals['signal'] = signals['signal'].ffill()
    signals['signal'] = signals['signal'].fillna(0)
    return signals

## 6. Backtest Engine

Dollar-neutral pairs backtest with:
- Rolling hedge ratio (no lookahead bias — `shift(1)` on beta)
- 10bps transaction cost per leg
- 6% risk-free rate (India RBI repo rate)

In [ ]:
def compute_metrics(returns, signals, risk_free_rate=RISK_FREE_RATE):
    """
    Compute risk-adjusted performance metrics net of transaction costs.
    """
    # Deduct transaction costs on signal changes
    signal_changes = signals['signal'].diff().abs()
    transaction_costs = signal_changes.fillna(0) * TRANSACTION_COST
    returns_net = returns - transaction_costs

    # Sharpe (annualised, net of risk-free rate)
    excess = returns_net.mean() - risk_free_rate / 252
    sharpe = np.sqrt(252) * excess / returns_net.std()

    # Sortino (downside deviation across all observations)
    downside_diff = np.minimum(returns_net - 0, 0)
    downside_std = np.sqrt((downside_diff ** 2).mean())
    sortino = np.sqrt(252) * returns_net.mean() / downside_std

    # CAGR
    n_years = len(returns_net) / 252
    total_return = (1 + returns_net).prod()
    cagr = total_return ** (1 / n_years) - 1

    # Max drawdown and longest continuous drawdown duration
    cumulative = (1 + returns_net).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_dd = drawdown.min()
    in_drawdown = (drawdown < 0).astype(int)
    drawdown_groups = in_drawdown * (
        in_drawdown.groupby((in_drawdown != in_drawdown.shift()).cumsum()).cumcount() + 1
    )
    max_dd_duration = drawdown_groups.max()

    return {
        'sharpe': sharpe,
        'sortino': sortino,
        'cagr': cagr,
        'max_drawdown': max_dd,
        'max_drawdown_duration': max_dd_duration,
        'total_return': total_return
    }


def backtest_pair(pair, prices):
    """
    Full backtest for a single pair.
    - Rolling beta for hedge ratio (price levels)
    - Z-score signal generation (price level spread)
    - PnL computation (returns spread with lagged beta to avoid lookahead)
    """
    try:
        rolling_beta = compute_rolling_beta(prices[pair[1]], prices[pair[0]]).dropna()

        if len(rolling_beta) == 0:
            return None

        # Price level spread → for z-score
        price_spread = prices[pair[1]] - rolling_beta * prices[pair[0]]
        price_spread = price_spread.dropna().replace([np.inf, -np.inf], np.nan).dropna()

        if len(price_spread) < ROLLING_WINDOW:
            return None

        # Returns spread → for PnL (shift(1) on beta avoids lookahead bias)
        returns = prices.pct_change()
        spread_returns = returns[pair[1]] - rolling_beta.shift(1) * returns[pair[0]]

        # Signals from price spread z-score
        zscore = compute_zscore(price_spread)
        signals = generate_signals(zscore)
        strategy_returns = signals['signal'].shift(1) * spread_returns

        metrics = compute_metrics(strategy_returns, signals)

        return {
            'pair': f"{pair[0]}/{pair[1]}",
            'sharpe': round(metrics['sharpe'], 2),
            'sortino': round(metrics['sortino'], 2),
            'cagr': round(metrics['cagr'], 2),
            'max_drawdown': round(metrics['max_drawdown'], 2),
            'max_drawdown_duration': int(metrics['max_drawdown_duration']),
            'total_return': round(float(metrics['total_return']), 2)
        }
    except Exception as e:
        print(f"Error: {pair} — {e}")
        return None

## 7. Results & Analysis

In [ ]:
# Run backtest on all stable pairs
backtest_results = []
for _, row in stable_pairs.iterrows():
    pair = tuple(row['pair'].split('/'))
    result = backtest_pair(pair, ticker_prices)
    if result:
        backtest_results.append(result)

backtest_df = pd.DataFrame(backtest_results).sort_values('sharpe', ascending=False)
backtest_df.to_csv('backtest_results.csv', index=False)
print(backtest_df.to_string())

In [ ]:
# Plot equity curve for best pair
best_pair = tuple(backtest_df.iloc[0]['pair'].split('/'))
print(f"Best pair: {best_pair[0]} / {best_pair[1]}")

rolling_beta = compute_rolling_beta(ticker_prices[best_pair[1]], ticker_prices[best_pair[0]]).dropna()
price_spread = ticker_prices[best_pair[1]] - rolling_beta * ticker_prices[best_pair[0]]
price_spread = price_spread.dropna().replace([np.inf, -np.inf], np.nan).dropna()
returns = ticker_prices.pct_change()
spread_returns = returns[best_pair[1]] - rolling_beta.shift(1) * returns[best_pair[0]]
zscore = compute_zscore(price_spread)
signals = generate_signals(zscore)
strategy_returns = signals['signal'].shift(1) * spread_returns

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Spread
price_spread.plot(ax=axes[0], title=f'{best_pair[0]}/{best_pair[1]} — Price Spread')
axes[0].axhline(price_spread.mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

# Z-score with signal thresholds
zscore.plot(ax=axes[1], title='Z-Score')
for level, color in [(2, 'green'), (-2, 'green'), (0, 'red'), (3, 'orange'), (-3, 'orange')]:
    axes[1].axhline(level, color=color, linestyle='--', alpha=0.7)

# Equity curve
(1 + strategy_returns).cumprod().plot(ax=axes[2], title='Cumulative Returns (Net of Costs)')
axes[2].axhline(1, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('best_pair_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Strategy Summary
best = backtest_df.iloc[0]
print("=" * 50)
print("STRATEGY SUMMARY")
print("=" * 50)
print(f"Universe:               NIFTY 50 ({len(valid_tickers)} stocks after filter)")
print(f"Period:                 {START_DATE} to {END_DATE}")
print(f"Total pairs scanned:    {len(all_pairs)}")
print(f"Cointegrated pairs:     {len(cointegrated_pairs)}")
print(f"Pairs passing filters:  {len(stable_pairs)}")
print()
print(f"Best pair:              {best['pair']}")
print(f"Sharpe ratio:           {best['sharpe']}")
print(f"Sortino ratio:          {best['sortino']}")
print(f"CAGR:                   {best['cagr']:.1%}")
print(f"Max drawdown:           {best['max_drawdown']:.1%}")
print(f"Total return:           {best['total_return']:.2f}x")
print()
print("KEY FINDINGS:")
print("- NIFTY 50 large-cap pairs have long mean-reversion half-lives (120-480 days)")
print("- Only pairs with stable rolling beta (std < 0.2) produce reliable signals")
print("- Transaction costs (10bps/leg) significantly impact EOD strategy Sharpe")
print("- Smallcap universe likely has faster reversion — next research direction")